In [ ]:
import os
import json
from pathlib import Path
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

from pypdf import PdfReader
from docx import Document

In [ ]:
def extract_text(file_path):
    """
    Reads files and extracts text based off file format.
    Supported Formats: .txt, .md, .pdf, .docx
    """

    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find file: {file_path}")
    
    extension = path.suffix.lower()
    extracted_text = ""

    if extension in [".txt", ".md", ".csv"]:
        with open(path, "r", encoding = "utf-8") as file:
            extracted_text = file.read()
    
    elif extension == ".pdf":
        reader = PdfReader(path)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

    elif extension == ".docx":
        doc = Document(path)
        for paragraph in doc.paragraphs:
            extracted_text += paragraph.text + "\n"
    
    else:
        raise ValueError(f"Unsupported file type: {extension}")
    
    return extracted_text.strip()
    

In [ ]:
class FormResponses(BaseModel):
    """
    Format LLM's output
    Forces LLM to return a dictionary of answers and list of missing items
    """

    extracted_answers: dict[str, str] = Field(
        description = "Dictionary mapping the exact form question to the extracted answer."
    )

    missing_information: list[str] = Field(
        description = "List of exact questions from the prompt that could not be answered using the text."
    )

In [ ]:
class LLM_Agent:
    def __init__(self, model_name = "gemini-3.1-pro-preview", temperature = 0.0):
        self.client = genai.Client()
        self.model_name = model_name
        self.temperature = temperature


        self.system_instructions = (
            "You are a strict data management assistant. Your objective is to extract information "
            "from user-provided scientific documents and metadata to accurately answer form questions "
            "for Data Management Plans, ReadMes, and DOIs. "
            "Keep answers extremely concise and strictly technical. If the provided document does not contain "
            "the answer to a requested question, DO NOT GUESS. Instead, add that exact question to the "
            "missing_information list and move on."
        )

    def extract_data(self, questions_list, document_text, schema):
        """
        Feeds the form questions and raw document text to the LLM,
        forcing the output into the Pydantic schema.
        """

        prompt = f"QUESTIONS TO ANSWER:\n{questions_list}\n\nSOURCE DOCUMENT:\n{document_text}"

        config = types.GenerateContentConfig(
            system_instruction = self.system_instructions,
            temperature = self.temperature,
            response_mime_type = "application/json",
            response_schema = schema
        )

        response = self.client.models.generate_content(
            model = self.model_name,
            contents = prompt,
            config = config
        )

        return json.loads(response.text)